In [ ]:
# train_model_colab_corrected.py
#
# This script trains a face recognition model in a Google Colab environment.

import os
import random
import shutil
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
from google.colab import drive


In [ ]:
# ==============================================================================
# 1. GOOGLE DRIVE SETUP AND DATA EXTRACTION
# ==============================================================================

def setup_dataset():
    """Mounts Google Drive and extracts the dataset."""
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

    file_path_in_drive = '/content/drive/MyDrive/my_datasets/11-785-fall-20-homework-2-part-2.zip'
    destination_path_in_colab = '/content/'

    if os.path.exists(file_path_in_drive):
        print(f"File found at {file_path_in_drive}. Extracting...")
        shutil.unpack_archive(file_path_in_drive, destination_path_in_colab)
        print(f"File extracted to {destination_path_in_colab}")
    else:
        print(f"Error: File not found at {file_path_in_drive}. Please check the path.")
        return None, None

    dataset_directory = '/content'
    train_dir = os.path.join(dataset_directory, "classification_data", "train_data")
    val_dir = os.path.join(dataset_directory, "classification_data", "val_data")

    if not os.path.exists(train_dir) or not os.path.exists(val_dir):
        print("Error: Extracted dataset directory structure is not as expected.")
        return None, None

    return train_dir, val_dir


In [ ]:
# ==============================================================================
# 2. SETUP AND DEPENDENCIES
# ==============================================================================

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

def print_setup():
    """Prints the TensorFlow and GPU setup information."""
    print(f"TensorFlow version: {tf.__version__}")
    print(f"Keras version: {keras.__version__}")
    print(f"GPU {'is available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE'}")

In [ ]:
# ==============================================================================
# 3. BASE EMBEDDING MODEL
# ==============================================================================

def create_base_model(input_shape=(64, 64, 3), embedding_size=128):
    """
    Creates a lightweight base model for face embedding using MobileNetV2.
    """
    base_model = keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    outputs = keras.layers.Dense(embedding_size, name="embedding")(x)

    return keras.Model(inputs, outputs, name="embedding_model")


In [ ]:
# ==============================================================================
# 4. DATA PREPARATION
# ==============================================================================

def load_and_preprocess_image(path, img_size=(64, 64)):
    """Loads and preprocesses an image from a file path."""
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    return image

def create_triplet_dataset(image_dir, batch_size=32):
    """Generates a triplet dataset dynamically using tf.data."""
    all_person_folders = [os.path.join(image_dir, person) for person in os.listdir(image_dir)]
    anchor_paths = []
    positive_paths = []

    # Generate all possible anchor-positive pairs
    for person_folder in all_person_folders:
        images_in_folder = [os.path.join(person_folder, img) for img in os.listdir(person_folder)]
        if len(images_in_folder) < 2:
            continue

        for i in range(len(images_in_folder)):
            for j in range(i + 1, len(images_in_folder)):
                anchor_paths.append(images_in_folder[i])
                positive_paths.append(images_in_folder[j])

    random.shuffle(anchor_paths)
    random.shuffle(positive_paths)

    # Create separate datasets for anchor and positive paths and zip them
    anchor_ds = tf.data.Dataset.from_tensor_slices(anchor_paths)
    positive_ds = tf.data.Dataset.from_tensor_slices(positive_paths)
    dataset = tf.data.Dataset.zip((anchor_ds, positive_ds))

    all_person_ids = os.listdir(image_dir)
    person_image_map = {name: [os.path.join(image_dir, name, img) for img in os.listdir(os.path.join(image_dir, name))] for name in all_person_ids}

    # The `add_negative` function now accepts two distinct arguments
    def add_negative(anchor_path, positive_path):
        def _py_func(anchor_path_tensor):
            anchor_path = anchor_path_tensor.numpy().decode('utf-8')

            # Extract the person ID from the anchor path
            anchor_person = anchor_path.split(os.sep)[-2]

            # Find a person different from the anchor person
            negative_person_id = random.choice(list(set(all_person_ids) - {anchor_person}))

            # Get a random image from the negative person's folder
            negative_path = random.choice(person_image_map[negative_person_id])

            return tf.constant(negative_path)

        negative_path_tensor = tf.py_function(
            _py_func, [anchor_path], tf.string
        )
        negative_path_tensor.set_shape(())

        return anchor_path, positive_path, negative_path_tensor

    dataset = dataset.map(add_negative, num_parallel_calls=tf.data.AUTOTUNE)

    # Preprocess images
    dataset = dataset.map(
        lambda anchor_path, positive_path, negative_path: (
            load_and_preprocess_image(anchor_path),
            load_and_preprocess_image(positive_path),
            load_and_preprocess_image(negative_path)
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    ).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
# ==============================================================================
# 5. METRIC LEARNING (SIAMESE NETWORK)
# ==============================================================================

class SiameseModel(keras.Model):
    """A Keras Model subclass for training with triplet loss."""
    def __init__(self, embedding_model, margin=0.5, distance_metric='euclidean'):
        super().__init__()
        self.embedding = embedding_model
        self.margin = margin
        self.distance_metric = distance_metric
        self.loss_tracker = keras.metrics.Mean(name='loss')

    def call(self, inputs):
        anchor, positive, negative = inputs
        return (
            self.embedding(anchor),
            self.embedding(positive),
            self.embedding(negative)
        )

    def train_step(self, data):
        anchor, positive, negative = data
        with tf.GradientTape() as tape:
            anchor_embedding = self.embedding(anchor)
            positive_embedding = self.embedding(positive)
            negative_embedding = self.embedding(negative)

            if self.distance_metric == 'euclidean':
                ap_distance = tf.reduce_sum(tf.square(anchor_embedding - positive_embedding), axis=-1)
                an_distance = tf.reduce_sum(tf.square(anchor_embedding - negative_embedding), axis=-1)
            else:
                ap_distance = 1 - tf.reduce_sum(tf.nn.l2_normalize(anchor_embedding, axis=-1) * tf.nn.l2_normalize(positive_embedding, axis=-1), axis=-1)
                an_distance = 1 - tf.reduce_sum(tf.nn.l2_normalize(anchor_embedding, axis=-1) * tf.nn.l2_normalize(negative_embedding, axis=-1), axis=-1)

            loss = tf.maximum(ap_distance - an_distance + self.margin, 0.0)

        gradients = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    @property
    def metrics(self):
        return [self.loss_tracker]

def train_triplet_model(train_ds, val_ds, base_model):
    """Trains the siamese model with triplet loss."""
    siamese_model = SiameseModel(base_model)
    siamese_model.compile(optimizer=keras.optimizers.Adam(0.0001))
    print("\nTraining Siamese Model with Triplet Loss...")
    siamese_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=[
            keras.callbacks.ModelCheckpoint("siamese_model_best.weights.h5", save_best_only=True, save_weights_only=True, monitor='val_loss')
        ]
    )


In [ ]:
# ==============================================================================
# 6. CLASSIFICATION METHOD
# ==============================================================================

def create_classification_model(base_model, num_classes):
    """Creates a classification model with a dense head on top of the base model."""
    inputs = keras.Input(shape=(64, 64, 3))
    x = base_model(inputs)
    x = keras.layers.Dense(256, activation='relu')(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name="classification_model")

def train_classification_model(train_ds, val_ds, base_model, num_classes):
    """Trains the classification model."""
    classification_model = create_classification_model(base_model, num_classes)
    classification_model.compile(
        optimizer=keras.optimizers.Adam(0.0001),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    print("\nTraining Classification Model...")
    classification_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=[
            keras.callbacks.ModelCheckpoint("classification_model_best.weights.h5", save_best_only=True, save_weights_only=True, monitor='val_accuracy')
        ]
    )


In [ ]:

# ==============================================================================
# 7. EVALUATION
# ==============================================================================

def create_verification_dataset(base_data_dir, pairs_text_path, batch_size=32):
    """Creates a verification dataset from a pairs text file."""
    pairs = []
    labels = []
    with open(pairs_text_path, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            p1, p2, label = line.strip().split()
            pairs.append((os.path.join(base_data_dir, p1), os.path.join(base_data_dir, p2)))
            labels.append(int(label))

    dataset = tf.data.Dataset.from_tensor_slices(pairs)
    dataset = dataset.map(
        lambda p1, p2: (load_and_preprocess_image(p1), load_and_preprocess_image(p2)),
        num_parallel_calls=tf.data.AUTOTUNE
    ).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    ground_truth_labels = np.array(labels)
    return dataset, ground_truth_labels

def create_distance_model(embedding_model, distance_metric='euclidean'):
    """Creates a model that takes two images and outputs their distance."""
    input_a = keras.Input(shape=(64, 64, 3))
    input_b = keras.Input(shape=(64, 64, 3))
    embedding_a = embedding_model(input_a)
    embedding_b = embedding_model(input_b)

    if distance_metric == 'euclidean':
        distance = keras.layers.Lambda(lambda x: tf.reduce_sum(tf.square(x[0] - x[1]), axis=-1), name='euclidean_distance')([embedding_a, embedding_b])
    else:
        distance = keras.layers.Lambda(lambda x: 1 - tf.reduce_sum(tf.nn.l2_normalize(x[0], axis=-1) * tf.nn.l2_normalize(x[1], axis=-1), axis=-1), name='cosine_distance')([embedding_a, embedding_b])

    return keras.Model(inputs=[input_a, input_b], outputs=distance, name="distance_model")

def evaluate_models(triplet_weights_path, classification_weights_path, verification_ds, verification_labels):
    """Loads trained models and evaluates them on a verification dataset."""
    triplet_embedding = create_base_model()
    triplet_embedding.load_weights(triplet_weights_path, by_name=True)
    classification_embedding = create_base_model()
    classification_embedding.load_weights(classification_weights_path, by_name=True)

    triplet_distance_model = create_distance_model(triplet_embedding)
    classification_distance_model = create_distance_model(classification_embedding)

    triplet_preds = triplet_distance_model.predict(verification_ds)
    classification_preds = classification_distance_model.predict(verification_ds)

    triplet_auc = roc_auc_score(verification_labels, triplet_preds)
    classification_auc = roc_auc_score(verification_labels, classification_preds)

    print(f"\nTriplet Model ROC AUC: {triplet_auc:.4f}")
    print(f"Classification Model ROC AUC: {classification_auc:.4f}")

    # Plot ROC curves
    triplet_fpr, triplet_tpr, _ = roc_curve(verification_labels, triplet_preds)
    classification_fpr, classification_tpr, _ = roc_curve(verification_labels, classification_preds)

    plt.figure(figsize=(8, 6))
    plt.plot(triplet_fpr, triplet_tpr, label=f'Triplet Loss (AUC = {triplet_auc:.4f})')
    plt.plot(classification_fpr, classification_tpr, label=f'Classification (AUC = {classification_auc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve for Face Verification')
    plt.legend()
    plt.show()

In [ ]:
# ==============================================================================
# 8. GRAPHING RESULTS
# ==============================================================================

# Data for plotting FPR and TPR at optimal thresholds (including Classification)
models_for_fpr_tpr = ['Euclidean Distance', 'Cosine Similarity', 'Classification']
fpr_values_for_fpr_tpr = [
    siamese_euclidean_fpr[siamese_euclidean_best_threshold_idx],
    siamese_cosine_fpr[siamese_cosine_best_threshold_idx],
    classification_fpr[classification_best_threshold_idx] # Include classification FPR
]
tpr_values_for_fpr_tpr = [
    siamese_euclidean_tpr[siamese_euclidean_best_threshold_idx],
    siamese_cosine_tpr[siamese_cosine_best_threshold_idx],
    classification_tpr[classification_best_threshold_idx] # Include classification TPR
]

x_for_fpr_tpr = np.arange(len(models_for_fpr_tpr)) # the label locations
width = 0.2 # the width of the bars

# Plotting FPR and TPR for all models
fig_fpr_tpr, ax_fpr_tpr = plt.subplots(figsize=(12, 7))
rects2_fpr_tpr = ax_fpr_tpr.bar(x_for_fpr_tpr - width/2, fpr_values_for_fpr_tpr, width, label='Best FPR')
rects3_fpr_tpr = ax_fpr_tpr.bar(x_for_fpr_tpr + width/2, tpr_values_for_fpr_tpr, width, label='Best TPR')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax_fpr_tpr.set_ylabel('Value')
ax_fpr_tpr.set_title('Best FPR and TPR for Each Model at Optimal Threshold')
ax_fpr_tpr.set_xticks(x_for_fpr_tpr)
ax_fpr_tpr.set_xticklabels(models_for_fpr_tpr)
ax_fpr_tpr.legend()
ax_fpr_tpr.set_ylim([0, 1]) # FPR and TPR are between 0 and 1

# Add value labels on top of the bars
def autolabel(rects, ax_obj):
    """Attach a text label above each bar in *rects*, displaying its height."""
    for rect in rects:
        height = rect.get_height()
        ax_obj.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects2_fpr_tpr, ax_fpr_tpr)
autolabel(rects3_fpr_tpr, ax_fpr_tpr)

plt.show()


# Data for plotting Optimal Thresholds (excluding Classification)
models_thresholds = ['Euclidean Distance', 'Cosine Similarity']
thresholds_values = [
    siamese_euclidean_thresholds[siamese_euclidean_best_threshold_idx],
    siamese_cosine_thresholds[siamese_cosine_best_threshold_idx]
    # Exclude classification_thresholds[classification_best_threshold_idx]
]

x_thresholds = np.arange(len(models_thresholds)) # the label locations
width = 0.2 # the width of the bars

# Plotting Optimal Thresholds for Metric Learning models
fig_thresh, ax_thresh = plt.subplots(figsize=(8, 6))
rects1_thresh = ax_thresh.bar(x_thresholds, thresholds_values, width, label='Optimal Threshold')

ax_thresh.set_ylabel('Optimal Threshold Value')
ax_thresh.set_title('Optimal Threshold for Metric Learning Models')
ax_thresh.set_xticks(x_thresholds)
ax_thresh.set_xticklabels(models_thresholds)
ax_thresh.legend()

autolabel(rects1_thresh, ax_thresh)

plt.show()

print("\nOptimal Thresholds for Metric Learning Models:")
print(f"Euclidean Distance: {siamese_euclidean_thresholds[siamese_euclidean_best_threshold_idx]:.2f}")
print(f"Cosine Similarity: {siamese_cosine_thresholds[siamese_cosine_best_threshold_idx]:.2f}")

# Keep the ROC curve plot as it shows performance across all thresholds
plt.figure(figsize=(8, 6))
plt.plot(classification_fpr,    classification_tpr,    label=f"Classification (Traditional Classification) (AUC: {classification_auc:.2f})")
plt.plot(siamese_euclidean_fpr, siamese_euclidean_tpr, label=f"Euclidean Distance as loss function (Metric Learning) (AUC: {siamese_euclidean_auc:.2f})")
plt.plot(siamese_cosine_fpr,    siamese_cosine_tpr,    label=f"Cosine Distance as loss function (Metric Learning) (AUC: {siamese_cosine_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", label=f"baseline")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curves for Face Verification")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:


# ==============================================================================
# MAIN EXECUTION BLOCK
# ==============================================================================

if __name__ == '__main__':
    print_setup()

    train_dir, val_dir = setup_dataset()
    if not train_dir or not val_dir:
        print("Data setup failed. Exiting.")
    else:
        embedding_model = create_base_model()

        # Data for Triplet Training
        triplet_train_ds = create_triplet_dataset(train_dir)
        triplet_val_ds = create_triplet_dataset(val_dir)
        train_triplet_model(triplet_train_ds, triplet_val_ds, embedding_model)

        # Data for Classification Training
        cls_train_ds = create_classification_dataset(train_dir)
        cls_val_ds = create_classification_dataset(val_dir)
        num_classes = len(cls_train_ds.class_names)
        train_classification_model(cls_train_ds, cls_val_ds, embedding_model, num_classes)

        # Evaluation
        verification_ds, verification_labels = create_verification_dataset(
            '/content/11-785-fall-20-homework-2-part-2',
            os.path.join('/content/11-785-fall-20-homework-2-part-2', 'verification_pairs_val.txt')
        )
        evaluate_models("siamese_model_best.weights.h5", "classification_model_best.weights.h5", verification_ds, verification_labels)

TensorFlow version: 2.18.0
Keras version: 3.8.0
GPU is available
Mounting Google Drive...
Mounted at /content/drive
File found at /content/drive/MyDrive/my_datasets/11-785-fall-20-homework-2-part-2.zip. Extracting...
File extracted to /content/


/tmp/ipython-input-3728939312.py:66: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Training Siamese Model with Triplet Loss...
Epoch 1/10
455933/632878 ━━━━━━━━━━━━━━━━━━━━ 1:13:18 25ms/step - loss: 3.0436